In [1]:
# exploring the vessel data before deciding a matching strategy.
# imo should be the stable key, mmsi changes over time - want to see how dirty it is.
import pandas as pd
pd.set_option("display.max_columns", 60)

df = pd.read_csv("../data/case_study_dataset_202509152039.csv")
df.shape

(1734, 50)

In [2]:
df.head()

,imo,mmsi,name,aisClass,callsign,length,width,vessel_type,flag,deadweight,grossTonnage,builtYear,tpcmi,netTonnage,hullTypeCode,draught,lengthOverall,airDraught,depth,beamMoulded,berthCount,deadYear,shipBuilder,hullNumber,launchYear,mainEngineCount,mainEngineDesigner,propulsionType,engineDesignation,propellerCount,propellerType,staticData_updateTimestamp,last_position_accuracy,last_position_course,last_position_heading,last_position_latitude,last_position_longitude,last_position_maneuver,last_position_rot,last_position_speed,last_position_updateTimestamp,destination,draught.1,eta,matchedPort_latitude,matchedPort_longitude,matchedPort_name,matchedPort_unlocode,InsertDate,UpdateDate
0,1000000,212222100,XF,A,AAA,399.0,60.0,General Cargo,CY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-08-22 12:12:59.000,0.0,0.0,139.0,31.981463,120.020123,0.0,0.00,0.0,2025-08-22 12:12:59.000,AA,NaN,2022-09-12 00:00:00.000,NaN,NaN,NaN,NaN,2023-03-21 15:36:09.000,2025-09-15 10:48:32.000
1,9528574,636013854,MARCO,A,A8PX9,225.0,32.0,Dry Bulk,LR,81393.0,42708.0,2009.0,NaN,26526.0,NaN,14.41,225.00,NaN,20.00,32.26,NaN,NaN,Universal Maizuru,164,2009.0,1.0,MAN B&W,Conventional Fuel,7S50MC-C,1.0,NaN,2025-09-16 02:17:21.000,0.0,127.4,126.0,2.502393,101.506633,0.0,0.00,12.4,2025-09-16 03:10:25.000,SINGAPORE,13.1,2025-09-17 04:00:00.000,1.355364,103.824318,Singapore,SGSIN,2023-03-21 14:36:07.000,2025-09-16 03:15:32.000
2,9752709,563091700,ASIA INSPIRE,A,9V5666,180.0,30.0,Chemical Tanker,SG,34500.0,23200.0,2019.0,NaN,8961.0,DH,9.20,180.00,NaN,15.40,30.00,NaN,NaN,Taizhou Sanfu,SF140502,2018.0,1.0,Wartsila,Conventional Fuel,5RT-flex50D,NaN,NaN,2025-09-11 03:10:55.000,0.0,285.0,52.0,3.415000,112.998333,0.0,6.43,0.0,2025-09-16 02:23:52.000,MY BTU,7.5,2025-09-13 22:00:00.000,3.266667,113.066666,Bintulu,MYBTU,2023-03-21 11:32:44.000,2025-09-16 02:39:50.000
3,9297357,341887000,HAMMURABI,A,V4WC4,252.0,44.0,Crude Tanker,KN,104999.0,63462.0,2006.0,99.3,34210.0,DH,14.18,251.51,39.82,21.30,43.81,NaN,NaN,Samsung,1531,2005.0,1.0,B&W,Motor,7S60MC-C,1.0,NaN,2025-08-22 12:25:15.000,0.0,116.0,107.0,40.771517,29.482083,0.0,0.04,1.5,2025-08-22 12:34:59.000,PORT SAID,14.1,2023-05-17 16:00:00.000,31.266666,32.299999,Port Said,EGPSD,2023-03-21 12:35:48.000,2025-09-16 02:48:07.000
4,9544023,440307000,FC DELLA,A,D7IE,109.0,18.0,Chemical Tanker,KR,6525.0,4956.0,2009.0,16.7,2161.0,DH,7.00,107.75,32.23,8.75,18.25,NaN,NaN,Pha Rung,FM-05,2008.0,1.0,Hanshin,Conventional Fuel,LH46L,1.0,NaN,2025-09-11 18:38:41.000,1.0,242.9,240.0,34.638325,128.784570,0.0,0.00,12.1,2025-09-11 20:51:00.000,KRTSN,5.5,2025-09-13 13:30:00.000,37.000000,126.366669,Daesan,KRTSN,2023-05-18 01:58:50.000,2025-09-15 08:53:27.000


In [3]:
# nulls + cardinality per column to get a feel for what's usable.
# notes from this: imo only 734 unique / 1734 rows -> lots of repeats. mmsi all unique.
# registry fields (tonnage, builtYear, dims) missing >55%; AIS position mostly present.
# deadYear, propellerType 100% null -> ignore. and there are two 'draught' columns
# (pandas renamed the 2nd draught.1) - check that next.
pd.DataFrame({
    "nulls": df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(1),
    "nunique": df.nunique(),
})

,nulls,null_pct,nunique
imo,0,0.0,734
mmsi,0,0.0,1734
name,40,2.3,1436
aisClass,47,2.7,2
callsign,91,5.2,1341
length,58,3.3,254
width,58,3.3,75
vessel_type,70,4.0,59
flag,84,4.8,129
deadweight,1035,59.7,439


In [4]:
# two 'draught' columns - check the raw header to see where they sit
with open("../data/case_study_dataset_202509152039.csv") as f:
    header = [c.strip().strip('"') for c in f.readline().split(",")]
[(i, c) for i, c in enumerate(header) if c == "draught"]
# 15 is next to hullTypeCode/lengthOverall -> design draught (static)
# 42 is next to destination/eta -> AIS reported draught (how loaded right now)

[(15, 'draught'), (42, 'draught')]

In [5]:
# so they're different measurements, not a dup. confirm: where both exist, do they differ?
df = df.rename(columns={"draught": "draught_static", "draught.1": "draught_ais"})
both = df[["draught_static", "draught_ais"]].dropna()
(both["draught_static"] != both["draught_ais"]).mean(), len(both)
# ~97% differ -> keep both, don't drop. ais draught is usually < design (not always full)

(0.9699842022116903, 633)

In [6]:
# how bad is the imo? validate the checksum (7 digits, 7th = check digit) and count.
# check digit: sum of first 6 digits weighted 7,6,5,4,3,2, mod 10 == 7th digit.
def imo_valid(imo):
    s = str(int(imo))
    if len(s) != 7:
        return False
    total = 0
    for i in range(6):
        total += int(s[i]) * (7 - i)
    check_digit = total % 10
    return check_digit == int(s[6])

df["imo_valid"] = df["imo"].apply(imo_valid)
df["imo_valid"].value_counts()

imo_valid
True     1064
False     670
Name: count, dtype: int64

In [7]:
# 670 invalid is a lot... but what are they? look at the invalid values
df.loc[~df["imo_valid"], "imo"].value_counts().head(12)

imo
0            252
1000000      144
3395388      100
2097152       39
2097216       19
1234560       14
8000000       12
123456789     10
3407872        7
4194336        7
1179648        6
1111110        5
Name: count, dtype: int64

In [8]:
# right, 0 / 1000000 / 123456789 etc are placeholders reused across many rows.
# so "invalid" is mostly "IMO unknown", not typos. split them:
#   sentinel = invalid value that shows up on >1 row (filler)
#   dirty    = invalid value on a single row (real typo)
inv = df.loc[~df["imo_valid"], "imo"].value_counts()

sentinels = set()
for imo_value, n in inv.items():
    if n > 1:
        sentinels.add(imo_value)

n_sent = df["imo"].isin(sentinels).sum()
n_dirty = (~df["imo_valid"]).sum() - n_sent
print("sentinel/placeholder:", n_sent)
print("dirty (real typos)  :", n_dirty)

sentinel/placeholder: 617
dirty (real typos)  : 53


In [9]:
# can't just drop the invalid ones - are they still real ships? check other fields
inv_rows = df[~df["imo_valid"]]
inv_rows[["mmsi", "name", "length", "vessel_type", "flag"]].notna().mean().round(2)
# ~90%+ have name/dims/flag -> real vessels missing only the IMO key. need a fallback, not delete.

mmsi           1.00
name           0.96
length         0.94
vessel_type    0.91
flag           0.90
dtype: float64

In [10]:
# 734 unique imo but 1734 rows. why the dups? suspect mmsi changes, but the
# placeholders (imo 0 etc) inflate it too. first: same row copied, or different records?
print("fully identical rows :", df.duplicated().sum())
print("dup (imo, mmsi) pairs:", df.duplicated(subset=["imo", "mmsi"]).sum())
# both 0 -> not copies. each (imo, mmsi) is one snapshot. so dups come from different mmsi.

fully identical rows : 0
dup (imo, mmsi) pairs: 0


In [11]:
# look at valid IMOs only (placeholders would merge unrelated ships)
valid = df[df["imo_valid"]]
vc = valid["imo"].value_counts()
dup = vc[vc > 1]
print("valid IMOs seen >1 time:", len(dup))

# for those, is the duplication always a mmsi change?
g = valid[valid["imo"].isin(dup.index)].groupby("imo")
print("of those, how many have >1 distinct mmsi:", (g["mmsi"].nunique() > 1).sum())
# 196 / 196 -> every duplicated valid IMO is a mmsi change. clean result.

valid IMOs seen >1 time: 196
of those, how many have >1 distinct mmsi: 196


In [12]:
# eyeball one - same ship, mmsi (and flag) changed over time
valid[valid["imo"] == 9259264][
    ["imo", "mmsi", "name", "flag", "staticData_updateTimestamp"]
].sort_values("staticData_updateTimestamp")

,imo,mmsi,name,flag,staticData_updateTimestamp
898,9259264,538007248,SANMAR SONGBIRD,MH,2025-08-22 12:13:01.000
1437,9259264,419001774,SANMAR SONGBIRD,IN,2025-09-01 20:47:18.000


In [13]:
# careful though - merging by IMO isn't trivial. some "immutable" fields disagree
# within the same IMO (length can't change -> data error). e.g.:
valid[valid["imo"] == 1006568][["imo", "mmsi", "name", "flag", "length", "width"]]
# length 44 vs 47 -> need conflict resolution (majority/latest) when building the golden record

,imo,mmsi,name,flag,length,width
214,1006568,244650177,LADY DUVERA,NL,44.0,8.0
248,1006568,244076672,LADY DUVERA,NL,47.0,9.0
1151,1006568,224650177,LADY DUVERA,ES,47.0,9.0


takeaways for matching:
- imo is the key when valid (1064 rows), but ~617 placeholders + ~53 typos -> validate first
- dups among valid imo are purely mmsi change -> safe to merge by imo, keep mmsi as history
- mmsi unique but unstable -> not a key
- keyless rows (placeholder/dirty imo) are still real ships -> fuzzy match, don't drop
- merging needs conflict resolution: immutable fields (length) sometimes disagree
- keep draught_static and draught_ais separate

identity logic in 02_identity_resolution.ipynb